# Working with Arrays and Nested Data in SQL

This notebook shows you how to work with arrays, nested structures, and maps in SQL without having to explode them or write complicated code. These are super useful when you're dealing with JSON data or complex nested columns.

## What you'll see

**Arrays** - Use `TRANSFORM()` to change every item in an array at once
* Example: make all product names uppercase in one line

**Nested structures** - Work with arrays of objects (like product name + price)
* Example: calculate tax for each product in an order without unpacking everything

**Aggregations** - Use `AGGREGATE()` to sum up values across an array
* Example: get the total price of all items in an order

**Maps** - Use `TRANSFORM_VALUES()` to update all the values in a key-value map
* Example: apply a 10% markup to all product prices at once

## Why this matters

Instead of exploding arrays (which creates tons of rows), then aggregating them back, you can just transform them in place. It's faster and way easier to read.

## How to use it

Just run all the cells in order. Everything uses temporary views so there's no setup needed. Look at each example and see how the data changes.

These patterns come in handy when you're working with JSON files, API responses, or any data with nested structures.

In [0]:
Create or replace temp view tv_higher_order
as 
select * from 
values
(1,array('smartphone','laptop','tablet')),
(2,array('smartphone','laptop','tablet','watch')),
(3,array('smartphone','laptop','tablet','watch'))
as order(order_id,items)

In [0]:
select * from  tv_higher_order


In [0]:
select order_id,
transform(items, x -> UPPER(x))
from tv_higher_order


In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_smart_gadgets
AS
SELECT * from values
(1,
    ARRAY(
        NAMED_STRUCT('name', 'Smart watch', 'price', 250),
        NAMED_STRUCT('name', 'Smart phone', 'price', 690),
        named_struct('name', 'Smart TV', 'price', 450))),
    (2,ARRAY(
        NAMED_STRUCT('name', 'Fitness Tracker', 'price', 129),
        NAMED_STRUCT('name', 'Smartwatch', 'price', 399),
        Named_struct('name', 'Smart Speaker', 'price', 199))
) 
AS gadget_groups (order_id, items);

In [0]:
select * from tv_smart_gadgets


In [0]:
select order_id,
transform (items, x -> named_struct('name', Upper(x.name),
                                   'price', Round(x.price * 0.1, 2))) as items_with_tax
from tv_smart_gadgets

In [0]:
select order_id,
transform(items, x -> named_struct('name', Upper(x.name), 
                                  'price' , Round(x.price *0.1, 2))) as items_with_tax
from tv_smart_gadgets

In [0]:
select order_id,
aggregate(items, 0, (acc,x) -> acc + x.price) as total
from tv_smart_gadgets

In [0]:
Create or replace temp view tv_map_func
as 
select * from 
values
(1,map('phone',699,'ipad',1000)),
(1,map('apple',699,'watch',1000)),
(1,map('calculator',699,'computer',1000))
as order(order_id,items)

In [0]:
select * from tv_map_func

In [0]:
select order_id,
transform_values(items, (order_id, items) -> Round(items * 1.1,2 ))
from tv_map_func